# Statista Chat-to-Slides Generator

Takes a structured Statista AI chat JSON response and generates a PowerPoint deck (with an
optional PDF export) summarizing the analysis: a title slide, one chart slide per insight
category, and a closing summary slide. See `README.md` for setup instructions.

In [1]:
! pip install python-pptx

## Imports

(the `pip install` cell above is a convenience — if you've already run
`pip install -r requirements.txt` per the README, it's a no-op)

In [2]:
from slides_code.builder import DeckBuilder

In [3]:
import json
import os
import subprocess

## Load and inspect the input JSON

In [4]:
with open('Task Requirment/gen_z_purchase_behavior_analysis.json', 'r') as file:
    data = json.load(file)  

print(data.keys()) # top-level field names, inspecting structure

dict_keys(['analysis'])


In [5]:
# exploratory: full raw structure, not part of the actual pipeline
print(data)

{'analysis': {'title': 'Gen Z Purchase Behavior: Key Insights from Statista Data', 'summary': 'Regarding the purchasing behavior of Gen Z, the available Statista data presents the following picture:\n\nDigital impulses have a particularly strong influence on purchasing decisions: Influencer posts influenced 57% of Gen Z in the USA in 2025, and posts from brands they already follow motivated 54% to purchase a product or service. In the fashion context, product experience also plays an important role: 51% of Gen Z in the USA stated in 2025 that trying on clothing before purchase is essential. When it comes to brands, the available data shows a strong beauty orientation: in 2025, The Ordinary was the most popular brand among Gen Z in the USA, and beauty brands dominated the brands that are disproportionately popular with Gen Z. At the same time, there is evidence of brand loyalty: two out of five Gen Z consumers in the USA stated in 2025 that they have a favorite brand they shop from. For

## Build the deck

In [6]:
analysis = data['analysis']
analysis.keys()

dict_keys(['title', 'summary', 'key_insights', 'data_sources', 'methodology'])

In [7]:
db = DeckBuilder()

subtitle = f"Prepared from Statista AI Chat Data — {analysis['data_sources']['primary_region']}, 2025"
db.add_title_slide(analysis['title'], subtitle)

for insight in analysis['key_insights']:
    db.add_chart_slide(
        insight['category'],
        insight['insight'],
        insight['metrics'],
        insight['source']
    )

db.add_summary_slide(analysis['summary'], analysis['data_sources'].get('caveat'))
db.save('output/deck.pptx')

## Export to PDF (via LibreOffice, headless)

In [8]:
soffice_path = "/Applications/LibreOffice.app/Contents/MacOS/soffice"
result = subprocess.run(
    [soffice_path, "--headless", "--convert-to", "pdf", "--outdir", "output", "output/deck.pptx"],
    capture_output=True,
    text=True
)
print(result.stdout)
print("Return code:", result.returncode)

convert /Users/hafeez/Code/Statista-Case-Study/output/deck.pptx as a Impress document -> /Users/hafeez/Code/Statista-Case-Study/output/deck.pdf using filter : impress_pdf_Export

Return code: 0


## Verify output

In [9]:

for f in ["output/deck.pptx", "output/deck.pdf"]:
    if os.path.exists(f):
        size_kb = os.path.getsize(f) / 1024
        print(f"OK  {f}  ({size_kb:.1f} KB)")
    else:
        print(f"MISSING  {f}")

OK  output/deck.pptx  (52.2 KB)
OK  output/deck.pdf  (56.6 KB)
